# Backend API Test Notebook
Use this notebook to smoke-test FastAPI endpoints.

Set `API_BASE` in the next cell to either your Render URL or `http://localhost:3000` for local dev.

> **Note:** The Render free tier spins down after 15 minutes of inactivity. The first request after a cold start can take 30–60 seconds — just wait and retry.


In [6]:
import json
from typing import Any
import requests

LOCAL_URL = "http://localhost:3000"
RENDER_URL = "https://london-explorer.onrender.com"  # ← paste your Render URL here
TIMEOUT_SECONDS = 30

# Auto-select: use local if it's up, otherwise fall back to Render
try:
    requests.get(f"{LOCAL_URL}/health", timeout=2)
    API_BASE = LOCAL_URL
    print(f"✓ Local server detected — using {API_BASE}")
except requests.exceptions.ConnectionError:
    API_BASE = RENDER_URL
    print(f"✓ Local server not running — using {API_BASE}")


✓ Local server not running — using https://london-explorer.onrender.com


In [10]:


def call_api(path: str, params: dict[str, Any] | None = None) -> Any:
    url = f"{API_BASE}{path}"
    response = requests.get(url, params=params, timeout=TIMEOUT_SECONDS)
    try:
        response.raise_for_status()
    except requests.HTTPError as exc:
        detail = response.text
        raise requests.HTTPError(f"{exc}\nResponse body: {detail}") from exc
    return response.json()

def preview(payload: Any, max_items: int = 3):
    if isinstance(payload, dict) and "data" in payload and isinstance(payload["data"], list):
        data = payload["data"]
        summary = {k: v for k, v in payload.items() if k != "data"}
        print("Summary:")
        print(json.dumps(summary, indent=2))
        print("\nData preview:")
        print(json.dumps(data[:max_items], indent=2))
        print(f"\nData length: {len(data)}")
        return

    print(json.dumps(payload, indent=2))


## 1) Health Check

In [11]:
health = call_api("/health")
preview(health)

{
  "status": "ok"
}


## 2) Tiles Endpoint
Adjust the viewport/filter params as needed.
Use `res` directly: `7=city`, `8=neighbourhood`, `9=street`, `10=finest`.

In [12]:
tiles_params = {
    "sw_lat": 51.48,
    "sw_lng": -0.22,
    "ne_lat": 51.54,
    "ne_lng": -0.06,
    "res": 8,
    "cuisine": "",
    "cost": "",
    "venue_type": "",
    "score_basis": 0,
    "score_tier": 0,
}

tiles = call_api("/api/tiles", params=tiles_params)
preview(tiles)

Summary:
{
  "mode": "tiles",
  "resolution": 8
}

Data preview:
[
  {
    "tile": "88195da4d7fffff",
    "count": 154
  },
  {
    "tile": "88194ad145fffff",
    "count": 66
  },
  {
    "tile": "88194ad301fffff",
    "count": 182
  }
]

Data length: 212


## 3) Nearby Endpoint

In [13]:
nearby_params = {
    "lat": 51.5074,
    "lng": -0.1278,
    "radius_m": 1000,
    "cuisine": "",
    "cost": "",
    "venue_type": "",
    "score_basis": 0,
    "rank_threshold": 0,
    "page": 1,
}

nearby = call_api("/api/nearby", params=nearby_params)
preview(nearby)

Summary:
{
  "page": 1,
  "page_size": 80
}

Data preview:
[
  {
    "id": "ChIJbULaEJIFdkgRZW6RTml3Npw",
    "display_name": "Zylia",
    "lat": 51.51023610000001,
    "lon": -0.1242071,
    "cuisine_type": "Unspecified",
    "venue_type": "Dine-In",
    "cost": "40+",
    "rating": 5.0,
    "user_rating_count": 172,
    "operational": true,
    "rank": 0.9947295784950256
  },
  {
    "id": "ChIJnd4G6-YFdkgRlb589_lLHM0",
    "display_name": "Vasiniko\ud83c\udf55",
    "lat": 51.5114849,
    "lon": -0.1209963999999999,
    "cuisine_type": "Pizza",
    "venue_type": "Dine-In",
    "cost": "20+",
    "rating": 4.900000095367432,
    "user_rating_count": 6806,
    "operational": true,
    "rank": 0.9928964376449585
  },
  {
    "id": "ChIJp9exgRcFdkgRpjsnrbLsyto",
    "display_name": "Brother Marcus Covent Garden",
    "lat": 51.5127993,
    "lon": -0.1263622,
    "cuisine_type": "Mediterranean",
    "venue_type": "Dine-In",
    "cost": "20+",
    "rating": 4.900000095367432,
    "user_ra

## 4) Place Detail Endpoint
Run the next cell after running nearby/tiles so you can pick a real place id.

In [14]:
sample_place_id = nearby.get("data", [{}])[0].get("id") if isinstance(nearby, dict) else None
sample_place_id

'ChIJbULaEJIFdkgRZW6RTml3Npw'

In [15]:
if not sample_place_id:
    raise ValueError("No place id available. Set sample_place_id manually and retry.")

place = call_api(f"/api/place/{sample_place_id}")
preview(place)

{
  "id": "ChIJbULaEJIFdkgRZW6RTml3Npw",
  "display_name": "Zylia",
  "primary_type_display_name": "Restaurant",
  "rating": 5.0,
  "user_rating_count": 172,
  "short_formatted_address": "6 Bedford St, London",
  "google_maps_uri": "https://maps.google.com/?cid=11256315612832558693&g_mp=Cilnb29nbGUubWFwcy5wbGFjZXMudjEuUGxhY2VzLlNlYXJjaE5lYXJieRACGAQgAA",
  "website_uri": "https://zyliataverna.com/",
  "types": "['restaurant', 'food', 'point_of_interest', 'establishment']",
  "primary_type": "restaurant",
  "is_chain": false,
  "predicted_type": null,
  "cuisine_type": "Unspecified",
  "venue_type": "Dine-In",
  "lat": 51.51023610000001,
  "lon": -0.1242071,
  "h3_r10": "8a194ad324c7fff",
  "pcd": "WC2E 9HZ",
  "areacode": "WC2E",
  "wheelchair_access": null,
  "operational": true,
  "cost": "40+",
  "wilson_1": 0.9781530499458313,
  "normal_1": 0.9947295784950256,
  "tier": 4,
  "tier_d": 4,
  "tier_independent": 4
}
